# OFDM

> Symbols on subcarriers, each seeing its subcarrier's channel.

In [ ]:
#| default_exp waveform.ofdm

In [ ]:
#| hide
from nbdev.showdoc import *

`OFDM` lays the symbols on the subcarriers, filling them in turn, so each symbol sees its subcarrier's `h`: a
frequency-selective channel (e.g. `RayTracedPathsFading` with subcarriers) hits some symbols harder than others.

It is OFDM in the frequency domain, as Sionna's `ApplyOFDMChannel`: the cyclic prefix is taken to be long enough, so
there is no inter-symbol interference, and there are no pilots, guard bands or DC null: one row of subcarriers,
reused for as many OFDM symbols as the transmission needs. Sionna's full grid, `ResourceGrid` with its
`ResourceGridMapper` and pilots, would be this stage's next version, with a demapper as its receiving half and
channel estimation (see `equalization`): it needs `h` per OFDM symbol, which the fadings don't give yet.

In [ ]:
#| export
from __future__ import annotations

import math

import numpy as np

from comm_core.block import Block
from comm_core.waveform.single_carrier import _like

In [ ]:
#| export
class OFDM(Block):
    "OFDM in the frequency domain: the symbols fill the subcarriers in turn, each seeing its subcarrier's h; √P included."

    outputs = ('h_sym',)

    def call(
        self,
        x, # The symbols, (B, S)
        h, # h of each transmission at each subcarrier, (B, K)
        power, # Their transmit powers, (B,)
    ):
        "The channel of each symbol, √P included, (B, S): symbol s on subcarrier s mod K."
        n = x.shape[-1]
        return _like(np.stack([np.resize(h[b], n) * math.sqrt(float(power[b])) for b in range(len(h))]), x)

In [ ]:
h_sym = OFDM()(np.zeros((1, 5)), np.array([[1.0, 2j]]), np.array([4.0]))
assert h_sym.tolist() == [[2.0, 4j, 2.0, 4j, 2.0]]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()